In [0]:
import sys
sys.path.append("/Workspace/Users/ichibakry7@gmail.com/tfl-tracker/src")

from config import (
    BRONZE_ARRIVALS,
    BRONZE_DISRUPTIONS,
    BRONZE_STOP_POINTS,
    SILVER_ARRIVALS,
    SILVER_DISRUPTIONS,
    SILVER_STOP_POINTS,
)
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType, TimestampType
from datetime import datetime, timezone

# ── Setup ─────────────────────────────────────────────────
spark.sql("CREATE CATALOG IF NOT EXISTS tfl_pipeline")
spark.sql("CREATE SCHEMA IF NOT EXISTS tfl_pipeline.silver")
TRANSFORMED_AT = datetime.now(timezone.utc).isoformat()

def log(msg):
    print(f"[{datetime.now(timezone.utc).isoformat()}] {msg}")

In [0]:
# ─────────────────────────────────────────────────────────
# ARRIVALS
# ─────────────────────────────────────────────────────────
log("Starting arrivals transform...")

arrivals_raw = spark.table(BRONZE_ARRIVALS)

arrivals_silver = (
    arrivals_raw
    # Step 1: parse the JSON string into a struct so we can access fields
    .withColumn("payload", F.from_json(
        F.col("raw_payload"),
        schema="""
            id STRING,
            vehicleId STRING,
            naptanId STRING,
            stationName STRING,
            lineId STRING,
            lineName STRING,
            platformName STRING,
            direction STRING,
            destinationNaptanId STRING,
            destinationName STRING,
            expectedArrival STRING,
            timeToStation INT,
            timestamp STRING,
            modeName STRING
        """
    ))
    # Step 2: promote fields from the struct to top-level columns
    .select(
        F.col("payload.id").alias("arrival_id"),
        F.col("payload.vehicleId").alias("vehicle_id"),
        F.col("payload.naptanId").alias("naptan_id"),
        F.col("payload.stationName").alias("station_name"),
        F.col("payload.lineId").alias("line_id"),
        F.col("payload.lineName").alias("line_name"),
        F.col("payload.platformName").alias("platform"),
        F.col("payload.direction").alias("direction"),
        F.col("payload.destinationNaptanId").alias("destination_naptan"),
        F.col("payload.destinationName").alias("destination_name"),
        # Step 3: cast strings to proper types
        F.to_timestamp(F.col("payload.expectedArrival")).alias("expected_arrival"),
        F.col("payload.timeToStation").cast(IntegerType()).alias("time_to_station_sec"),
        F.to_timestamp(F.col("payload.timestamp")).alias("prediction_timestamp"),
        F.col("payload.modeName").alias("mode_name"),
        # Step 4: carry over pipeline metadata from Bronze
        F.to_timestamp(F.col("ingested_at")).alias("ingested_at"),
        F.col("run_id"),
    )
    # Step 5: drop rows where the essential identifiers are missing
    .filter(
        F.col("arrival_id").isNotNull() &
        F.col("vehicle_id").isNotNull() &
        F.col("naptan_id").isNotNull()
    )
)

(arrivals_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_ARRIVALS))

log(f"Arrivals: wrote {arrivals_silver.count()} records to Silver")

In [0]:
# ─────────────────────────────────────────────────────────
# DISRUPTIONS
# ─────────────────────────────────────────────────────────
log("Starting disruptions transform...")

disruptions_raw = spark.table(BRONZE_DISRUPTIONS)

disruptions_silver = (
    disruptions_raw
    .withColumn("payload", F.from_json(
        F.col("raw_payload"),
        schema="""
            category STRING,
            type STRING,
            categoryDescription STRING,
            description STRING,
            closureText STRING,
            affectedRoutes ARRAY<STRING>,
            affectedStops ARRAY<STRING>
        """
    ))
    .select(
        F.col("line_id"),
        F.col("payload.category").alias("category"),
        F.col("payload.type").alias("disruption_type"),
        # Clean the description — trim whitespace, collapse multiple spaces
        F.regexp_replace(
            F.trim(F.col("payload.description")), "\\s+", " "
        ).alias("description"),
        F.col("payload.closureText").alias("closure_text"),
        # Convert arrays to booleans — we care whether they exist, not their contents
        (F.size(F.col("payload.affectedRoutes")) > 0).alias("has_affected_routes"),
        (F.size(F.col("payload.affectedStops")) > 0).alias("has_affected_stops"),
        F.to_timestamp(F.col("ingested_at")).alias("ingested_at"),
        F.col("run_id"),
    )
    .filter(F.col("line_id").isNotNull())
)

(disruptions_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_DISRUPTIONS))

log(f"Disruptions: wrote {disruptions_silver.count()} records to Silver")

In [0]:
# ─────────────────────────────────────────────────────────
# STOP POINTS
# ─────────────────────────────────────────────────────────
log("Starting stop points transform...")

stops_raw = spark.table(BRONZE_STOP_POINTS)

# Helper: extract a value from additionalProperties by key
# additionalProperties is an array of {category, key, value} objects
# We filter to the matching key and pull the value out
def extract_prop(key: str):
    filtered = F.transform(
        F.filter(
            F.col("payload.additionalProperties"),
            lambda x: x["key"] == key
        ),
        lambda x: x["value"]
    )
    return F.when(F.size(filtered) > 0, F.element_at(filtered, 1))

stops_silver = (
    stops_raw
    .withColumn("payload", F.from_json(
        F.col("raw_payload"),
        schema="""
            naptanId STRING,
            commonName STRING,
            stopType STRING,
            status BOOLEAN,
            lat DOUBLE,
            lon DOUBLE,
            additionalProperties ARRAY<STRUCT<category: STRING, key: STRING, value: STRING>>
        """
    ))
    .select(
        F.col("payload.naptanId").alias("naptan_id"),
        F.col("payload.commonName").alias("common_name"),
        F.col("payload.stopType").alias("stop_type"),
        F.col("payload.lat").cast(DoubleType()).alias("lat"),
        F.col("payload.lon").cast(DoubleType()).alias("lon"),
        F.col("payload.status").alias("is_active"),
        # Extract individual properties from the additionalProperties array
        extract_prop("Zone").alias("zone"),
        (extract_prop("WiFi") == "yes").alias("has_wifi"),
        F.when(
            # 1. Check for explicit negative words
            F.lower(extract_prop("Lifts")).rlike("no|none|0|unavailable"), 
            False
        ).when(
            # 2. Check for explicit positive words or descriptions
            F.lower(extract_prop("Lifts")).rlike("yes|1|lift|access|available"), 
            True
        ).otherwise(
            # 3. If the key was missing or the text is weird, assume False
            False 
        ).alias("has_lifts"),
        # (F.coalesce(extract_prop("Lifts"), F.lit("")) != "").alias("has_lifts"),
        (extract_prop("Toilets") != "no").alias("has_toilets"),
        (extract_prop("Car park") == "yes").alias("has_car_park"),
        (extract_prop("Waiting Room") == "yes").alias("has_waiting_room"),
        F.trim(extract_prop("Help Points")).alias("help_points"),
        F.to_timestamp(F.col("ingested_at")).alias("ingested_at"),
        F.col("run_id"),
    )
    .filter(
        F.col("naptan_id").isNotNull() &
        (F.col("lat") != 0.0) &   # drop child platform records with no coordinates
        (F.col("lon") != 0.0)
    )
)

(stops_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_STOP_POINTS))

log(f"Stop points: wrote {stops_silver.count()} records to Silver")
log("Silver transform complete.")